In [ ]:
pip install torch transformers datasets accelerate scikit-learn pandas numpy

In [ ]:
import torch

# Verify CUDA availability
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_LABELS = 3
NUM_FOLDS = 5
BATCH_SIZE = 32
LEARNING_RATE = 3e-5
EPOCHS = 3
OUTPUT_DIR = "./results"
BEST_MODEL_DIR = "./models/best_model"

In [ ]:
def clean_text(text):
    """Cleans basic text noise like extra spaces, user handles, or broken links."""
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Loading tweet_eval dataset...")
dataset = load_dataset("tweet_eval", "sentiment")

# Combine splits for Stratified Cross-Validation
df_train = pd.DataFrame(dataset["train"])
df_val = pd.DataFrame(dataset["validation"])
df_test = pd.DataFrame(dataset["test"])
full_df = pd.concat([df_train, df_val, df_test], ignore_index=True)

# Apply clean_text method to dataset
full_df["text"] = full_df["text"].apply(clean_text)

# Tokenizer initialization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Preview the cleaned data
full_df.head()

In [ ]:
def compute_metrics(eval_pred):
    """Computes Accuracy, Precision, Recall, and Macro F1-Score"""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "macro_f1": f1}

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = {key: torch.tensor(val) if not isinstance(val, torch.Tensor) else val 
                          for key, val in encodings.items()}
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42) 

best_macro_f1 = -1.0
oof_predictions = np.zeros((len(full_df), NUM_LABELS)) 
oof_labels = np.zeros(len(full_df))

print(f"Starting {NUM_FOLDS}-Fold Stratified Cross-Validation...") 

for fold, (train_idx, val_idx) in enumerate(skf.split(full_df["text"], full_df["label"])):
    print(f"\n--- Training Fold {fold + 1}/{NUM_FOLDS} ---")
    
    # Instantiate fresh model per fold 
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

    model.to(device)
    
    # Tokenize splits
    train_encodings = tokenizer(list(full_df["text"].iloc[train_idx]), padding="max_length", truncation=True, max_length=MAX_LENGTH) 
    val_encodings = tokenizer(list(full_df["text"].iloc[val_idx]), padding="max_length", truncation=True, max_length=MAX_LENGTH)
    
    train_dataset = SentimentDataset(train_encodings, full_df["label"].iloc[train_idx].values)
    val_dataset = SentimentDataset(val_encodings, full_df["label"].iloc[val_idx].values)
    
    training_args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/fold_{fold}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE, 
        per_device_train_batch_size=BATCH_SIZE, 
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, 
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        fp16=torch.cuda.is_available()
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )
    
    trainer.train()

    
    # Track Out-of-Fold (OOF) predictions 
    predictions_output = trainer.predict(val_dataset)
    oof_predictions[val_idx] = predictions_output.predictions
    oof_labels[val_idx] = full_df["label"].iloc[val_idx].values
    
    # Save the absolute best iteration to the deployment directory 
    current_macro_f1 = trainer.evaluate()["eval_macro_f1"]
    if current_macro_f1 > best_macro_f1:
        best_macro_f1 = current_macro_f1
        os.makedirs(BEST_MODEL_DIR, exist_ok=True) 
        trainer.save_model(BEST_MODEL_DIR) 
        tokenizer.save_pretrained(BEST_MODEL_DIR) 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

print("=== FINAL GLOBAL EVALUATION REPORT ===")
oof_preds_labels = np.argmax(oof_predictions, axis=1)

# 1. Compute Aggregated Metrics
final_precision, final_recall, final_f1, _ = precision_recall_fscore_support(oof_labels, oof_preds_labels, average="macro") 
final_acc = accuracy_score(oof_labels, oof_preds_labels)

print(f"Aggregated Accuracy:  {final_acc:.4f}")
print(f"Aggregated Precision: {final_precision:.4f}")
print(f"Aggregated Recall:    {final_recall:.4f}")
print(f"Aggregated Macro F1:  {final_f1:.4f}")
print("\n" + "="*30)

# 2. Generate Confusion Matrix
cm = confusion_matrix(oof_labels, oof_preds_labels) 
class_names = ["Negative", "Neutral", "Positive"]

# 3. Plot Beautiful Heatmap using Seaborn
plt.figure(figsize=(8, 6))
sns.set_theme(style="whitegrid") 

sns.heatmap(
    cm, 
    annot=True,          # Write the raw numbers in each cell
    fmt="d",             # Format numbers as integers instead of scientific notation
    cmap="Blues",        
    xticklabels=class_names, 
    yticklabels=class_names,
    cbar=True,
    annot_kws={"size": 14, "weight": "bold"}
)

# Titles and Labels
plt.title("Aggregated Confusion Matrix (Out-of-Fold Predictions)", fontsize=16, pad=20, weight="bold")
plt.xlabel("Predicted Labels", fontsize=12, labelpad=10, weight="bold")
plt.ylabel("Actual Labels", fontsize=12, labelpad=10, weight="bold")

# Clean layout adjustments
plt.tight_layout()
plt.show()